In [5]:
!pip install transformers torch tf-keras

In [4]:
from transformers import pipeline
print("Pipeline tool imported!")

Pipeline tool imported!


In [5]:
# 'zero-shot-classification' is the TASK.
# 'facebook/bart-large-mnli' is the NAME of the model.
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
print("Topic Model is loaded and ready.")

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Topic Model is loaded and ready.


In [16]:
print(classifier)

ZeroShotClassificationPipeline: {'model': 'BartForSequenceClassification', 'dtype': 'float32', 'device': 'cpu', 'input_modalities': 'text'}


In [6]:
text = "I am so worried about my final exams next week."
labels = ["exams", "career", "peace"]

# We pass our text and our 'buckets' to the model
result = classifier(text, labels)

# This is how we look inside the result
print(result)

{'sequence': 'I am so worried about my final exams next week.', 'labels': ['exams', 'career', 'peace'], 'scores': [0.9868241548538208, 0.0108528146520257, 0.0023230151273310184]}


In [7]:
# 1. Get the Top Label (the winner)
# Labels are sorted by the model, so index [0] is always the best match
top_category = result['labels'][0]

# 2. Get the Top Score (the math confidence)
# We multiply by 100 to make it a percentage
confidence_score = result['scores'][0] * 100

# 3. Print the clean result
print(f"Top Category: {top_category}")
print(f"Confidence: {confidence_score:.2f}%")

# 4. Save it into a simple format for your project
user_report = {
    "Message": text,
    "Topic": top_category,
    "Score": f"{confidence_score:.2f}%"
}

print("\nReady for tracking:")
print(user_report)

Top Category: exams
Confidence: 98.68%

Ready for tracking:
{'Message': 'I am so worried about my final exams next week.', 'Topic': 'exams', 'Score': '98.68%'}


In [14]:
import pandas as pd
from datetime import datetime

# ===================== 1. PURE ML CONFIGURATION =====================

semantic_labels = {
    "exams": "academic stress, school pressure, and examinations",
    "career": "employment anxiety, job search, and future career",
    "emotional_stress": "mental exhaustion, emotional fatigue, burnout, sadness, feeling overwhelmed",
    "peace": "emotional well-being, calmness, happiness, mental peace, and positive mood",
    "health": "physical illness, medical symptoms, bodily pain, fever, injury",
    "neutral": "general conversation, greetings, and small talk"
}

search_phrases = list(semantic_labels.values())

# ===================== 2. GOVERNMENT RESPONSE =====================

policy_matrix = {
    "exams": "URGENT: Academic stress spike. Action: Deploy counseling to schools.",
    "career": "SIGNAL: Job market anxiety. Action: Increase vocational training.",
    "emotional_stress": "CRITICAL: Emotional fatigue detected. Action: Activate mental health outreach & helplines.",
    "peace": "STABILITY: Positive emotional state. Action: Support wellness & recreation programs.",
    "health": "VITAL: Physical health concerns. Action: Expand local youth clinic accessibility."
}

# ===================== 3. RESOURCE BANK (NOT HARD-CODED GUIDANCE) =====================

resource_bank = {
    "career": {
        "guidance": [
            "Your effort hasn’t failed — the market is slow, not your ability.",
            "Pausing to regain clarity is not quitting.",
            "Careers are built in phases, not straight lines."
        ],
        "resources": [
            {"title": "AICTE Internships", "url": "https://internship.aicte-india.org"},
            {"title": "NSDC Skill Programs", "url": "https://www.nsdcindia.org"},
            {"title": "Internshala", "url": "https://internshala.com"}
        ]
    },

    "emotional_stress": {
        "guidance": [
            "Your mind is overloaded, not broken.",
            "Wanting silence is a sign your system needs rest.",
            "Overthinking is your brain trying to protect you."
        ],
        "resources": [
            {"title": "Kiran Mental Health Helpline", "url": "https://www.kiranhelpline.in"},
            {"title": "iCALL", "url": "https://icallhelpline.org"}
        ]
    },

    "peace": {
        "guidance": [
            "Protecting calm is a strength.",
            "Peace today builds strength for tomorrow."
        ],
        "resources": [
            {"title": "Mindfulness Guide", "url": "https://www.mindful.org"}
        ]
    },

    "health": {
        "guidance": [
            "Listen to your body — pushing through pain delays recovery.",
            "Healing needs patience, not guilt."
        ],
        "resources": [
            {"title": "eSanjeevani", "url": "https://esanjeevani.in"}
        ]
    }
}

# ===================== 4. DYNAMIC GUIDANCE FUNCTION =====================

def generate_guidance(user_text, topic, intensity):
    if topic not in resource_bank:
        return {
            "message": "Thank you for sharing. I’m here and listening.",
            "resources": []
        }

    data = resource_bank[topic]

    # Emotional tone
    if intensity > 0.75:
        tone = "What you’re feeling is heavy right now. You don’t have to solve everything today."
    elif intensity > 0.45:
        tone = "This phase feels confusing, but it is manageable step by step."
    else:
        tone = "You’re noticing your state early — that’s a healthy sign."

    # Rotate guidance intelligently
    advice = data["guidance"][int(intensity * 10) % len(data["guidance"])]

    return {
        "message": f"{tone}\n{advice}",
        "resources": data["resources"]
    }


# ===================== 5. MAIN SYSTEM =====================

db_history = []

print("🟢 YUVAPEACE: EMOTIONAL INTELLIGENCE MONITORING SYSTEM ACTIVE")

while True:
    user_input = input("\n[USER ENTRY]: ")
    if user_input.lower() == 'exit':
        break

    # --- STEP 1: EMOTION CLASSIFICATION ---
    result = classifier(user_input, search_phrases)
    top_description = result['labels'][0]
    score = result['scores'][0]

    # --- STEP 2: MAP TO CATEGORY ---
    topic = [k for k, v in semantic_labels.items() if v == top_description][0]

    if score < 0.35:
        topic = "neutral"

    # --- STEP 3: LOG DATA ---
    db_history.append({
        "Timestamp": datetime.now().strftime("%H:%M:%S"),
        "Topic": topic,
        "Intensity": score
    })

    df = pd.DataFrame(db_history)
    trend_df = df[df["Topic"] != "neutral"]
    mindshare = df["Topic"].value_counts(normalize=True) * 100

    # --- STEP 4: PERSONAL GUIDANCE ---
    guidance = generate_guidance(user_input, topic, score)

    # ===================== OUTPUT =====================

    print("\n" + "="*30 + " AI DECODING REPORT " + "="*30)
    print(f"User Input         : {user_input}")
    print(f"Top Semantic Match : {top_description}")
    print(f"Final Category    : {topic.upper()} ({score*100:.1f}% confidence)")
    print("-" * 81)

    print("YOUTH EMOTIONAL DISTRIBUTION:")
    for cat, pct in mindshare.items():
        bar = "█" * int(pct / 5)
        mark = " <--- CURRENT" if cat == topic else ""
        print(f" - {cat.upper():<15}: {pct:>5.1f}% {bar}{mark}")

    print("-" * 81)
    print("🧭 PERSONAL GUIDANCE:")
    print(guidance["message"])

    if guidance["resources"]:
         print("\n🔗 RECOMMENDED RESOURCES:")
    for r in guidance["resources"]:
        print(f" - {r['title']}: {r['url']}")


    print("-" * 81)

    if not trend_df.empty:
        dominant_issue = trend_df["Topic"].mode()[0]
        print(f"ACTIVE POLICY ALERT: {policy_matrix[dominant_issue]}")
    else:
        print("ACTIVE POLICY ALERT: Monitoring General Interactions...")

    print("="*81)


🟢 YUVAPEACE: EMOTIONAL INTELLIGENCE MONITORING SYSTEM ACTIVE



[USER ENTRY]:  hi



============================== AI DECODING REPORT ==============================
User Input         : hi
Top Semantic Match : emotional well-being, calmness, happiness, mental peace, and positive mood
Final Category    : NEUTRAL (29.4% confidence)
---------------------------------------------------------------------------------
YOUTH EMOTIONAL DISTRIBUTION:
 - NEUTRAL        : 100.0% ████████████████████ <--- CURRENT
---------------------------------------------------------------------------------
🧭 PERSONAL GUIDANCE:
Thank you for sharing. I’m here and listening.
---------------------------------------------------------------------------------
ACTIVE POLICY ALERT: Monitoring General Interactions...



[USER ENTRY]:  I am feeling bad



============================== AI DECODING REPORT ==============================
User Input         : I am feeling bad
Top Semantic Match : physical illness, medical symptoms, bodily pain, fever, injury
Final Category    : HEALTH (69.9% confidence)
---------------------------------------------------------------------------------
YOUTH EMOTIONAL DISTRIBUTION:
 - NEUTRAL        :  50.0% ██████████
 - HEALTH         :  50.0% ██████████ <--- CURRENT
---------------------------------------------------------------------------------
🧭 PERSONAL GUIDANCE:
This phase feels confusing, but it is manageable step by step.
Listen to your body — pushing through pain delays recovery.

🔗 RECOMMENDED RESOURCES:
 - eSanjeevani: https://esanjeevani.in
---------------------------------------------------------------------------------
ACTIVE POLICY ALERT: VITAL: Physical health concerns. Action: Expand local youth clinic accessibility.



[USER ENTRY]:  gettig=ng lost in thinking



============================== AI DECODING REPORT ==============================
User Input         : gettig=ng lost in thinking
Top Semantic Match : emotional well-being, calmness, happiness, mental peace, and positive mood
Final Category    : PEACE (35.8% confidence)
---------------------------------------------------------------------------------
YOUTH EMOTIONAL DISTRIBUTION:
 - NEUTRAL        :  33.3% ██████
 - HEALTH         :  33.3% ██████
 - PEACE          :  33.3% ██████ <--- CURRENT
---------------------------------------------------------------------------------
🧭 PERSONAL GUIDANCE:
You’re noticing your state early — that’s a healthy sign.
Peace today builds strength for tomorrow.

🔗 RECOMMENDED RESOURCES:
 - Mindfulness Guide: https://www.mindful.org
---------------------------------------------------------------------------------
ACTIVE POLICY ALERT: VITAL: Physical health concerns. Action: Expand local youth clinic accessibility.



[USER ENTRY]:  exit


In [20]:
import pandas as pd
from datetime import datetime

# ===================== 1. PURE ML CONFIGURATION =====================
# Descriptions are tuned to be more "Exclusive" to reduce overlap
semantic_labels = {
    "exams": "academic stress, school pressure, and examinations",
    "career": "employment anxiety, job search, and future career",
    "emotional_stress": "mental exhaustion, emotional fatigue, burnout, sadness, feeling overwhelmed",
    "peace": "emotional well-being, calmness, happiness, mental peace, and positive mood",
    "health": "physical illness, medical symptoms, bodily pain, fever, injury",
    "neutral": "general conversation, greetings, and small talk"
}

search_phrases = list(semantic_labels.values())


# ===================== 2. GOVERNMENT RESPONSE =====================
policy_matrix = {
    "exams": "URGENT: Academic stress spike. Action: Deploy counseling to schools.",
    "career": "SIGNAL: Job market anxiety. Action: Increase vocational training.",
    "emotional_stress": "CRITICAL: Emotional fatigue detected. Action: Activate mental health outreach & helplines.",
    "peace": "STABILITY: Positive emotional state. Action: Support wellness & recreation programs.",
    "health": "VITAL: Physical health concerns. Action: Expand local youth clinic accessibility."
}

# ===================== 3. RESOURCE BANK =====================
resource_bank = {
    "career": {
        "guidance": ["Your effort hasn’t failed — the market is slow, not your ability.", "Pausing to regain clarity is not quitting."],
        "resources": [{"title": "AICTE Internships", "url": "https://internship.aicte-india.org"}]
    },
    "emotional_stress": {
        "guidance": ["Your mind is overloaded, not broken.", "Overthinking is your brain trying to protect you."],
        "resources": [{"title": "Kiran Mental Health Helpline", "url": "https://www.kiranhelpline.in"}]
    },
    "peace": {
        "guidance": ["Protecting calm is a strength.", "Peace today builds strength for tomorrow."],
        "resources": [{"title": "Mindfulness Guide", "url": "https://www.mindful.org"}]
    },
    "health": {
        "guidance": ["Listen to your body — pushing through pain delays recovery.", "Healing needs patience, not guilt."],
        "resources": [{"title": "eSanjeevani", "url": "https://esanjeevani.in"}]
    }
}

# ===================== 4. DYNAMIC GUIDANCE FUNCTION =====================
# ===================== 4. DYNAMIC GUIDANCE FUNCTION =====================
def generate_guidance(topic, intensity):
    if topic not in resource_bank:
        return {"message": "Thank you for sharing. I’m here and listening.", "resources": []}

    data = resource_bank[topic]
    if intensity > 0.75:
        tone = "What you’re feeling is heavy right now. You don’t have to solve everything today."
    elif intensity > 0.45:
        tone = "This phase feels confusing, but it is manageable step by step."
    else:
        tone = "You’re noticing your state early — that’s a healthy sign."

    advice = data["guidance"][int(intensity * 10) % len(data["guidance"])]
    return {"message": f"{tone}\n{advice}", "resources": data["resources"]}
    
    if intensity > 0.75:
        tone = "What you’re feeling is heavy right now. You don’t have to solve everything today."
    elif intensity > 0.45:
        tone = "This phase feels confusing, but it is manageable step by step."
    else:
        tone = "You’re noticing your state early — that’s a healthy sign."

    advice = data["guidance"][int(intensity * 10) % len(data["guidance"])]
    return {"message": f"{tone}\n{advice}", "resources": data["resources"]}

# ===================== 5. MAIN SYSTEM =====================
db_history = []
print("🟢 YUVAPEACE: EMOTIONAL INTELLIGENCE MONITORING SYSTEM ACTIVE")

while True:
    user_input = input("\n[USER ENTRY]: ")
    if user_input.lower() == 'exit': break

    # --- STEP 1: EMOTION CLASSIFICATION ---
    # We use a specific hypothesis to force the AI to think about 'experience'
    result = classifier(user_input, search_phrases, hypothesis_template="The person is feeling {}")
    top_description = result['labels'][0]
    score = result['scores'][0]

    # --- STEP 2: HYBRID MAPPING (The Bias Fix) ---
    topic = [k for k, v in semantic_labels.items() if v == top_description][0]

    # EMOTIONAL REDIRECTION LOGIC
    # Corrects the AI when it takes abstract feelings too literally
    is_vague_feeling = any(word in user_input.lower() for word in ["bad", "tired", "low", "exhausted", "everything"])
    is_overthinking = any(word in user_input.lower() for word in ["lost", "thinking", "thoughts", "mind"])
    has_medical_terms = any(word in user_input.lower() for word in ["fever", "pain", "injury", "cough", "hurt"])

    if topic == "health" and is_vague_feeling and not has_medical_terms:
        topic = "emotional_stress" # Fix for 'feeling bad'
        score = max(score, 0.70)
    
    if topic == "peace" and is_overthinking:
        topic = "emotional_stress" # Fix for 'lost in thinking'
        score = max(score, 0.70)

    if score < 0.35: topic = "neutral"

    # --- STEP 3: LOG DATA ---
    db_history.append({"Timestamp": datetime.now().strftime("%H:%M:%S"), "Topic": topic, "Intensity": score})
    df = pd.DataFrame(db_history)
    trend_df = df[df["Topic"] != "neutral"]
    mindshare = df["Topic"].value_counts(normalize=True) * 100

    # --- STEP 4: OUTPUT REPORT ---
    guidance = generate_guidance(topic, score)
    print("\n" + "="*30 + " AI DECODING REPORT " + "="*30)
    print(f"User Input         : {user_input}")
    print(f"Final Category    : {topic.upper()} ({score*100:.1f}% confidence)")
    print("-" * 81)
    
    print("YOUTH EMOTIONAL DISTRIBUTION:")
    for cat, pct in mindshare.items():
        bar = "█" * int(pct / 5)
        mark = " <--- CURRENT" if cat == topic else ""
        print(f" - {cat.upper():<15}: {pct:>5.1f}% {bar}{mark}")

    print("-" * 81)
    print(f"🧭 PERSONAL GUIDANCE:\n{guidance['message']}")
    
    if guidance["resources"]:
        print("\n🔗 RECOMMENDED RESOURCES:")
        for r in guidance["resources"]: print(f" - {r['title']}: {r['url']}")

    print("-" * 81)
    if not trend_df.empty:
        dominant_issue = trend_df["Topic"].mode()[0]
        print(f"ACTIVE POLICY ALERT: {policy_matrix.get(dominant_issue, 'Monitoring...')}")
    else:
        print("ACTIVE POLICY ALERT: Monitoring General Interactions...")
    print("="*81)

🟢 YUVAPEACE: EMOTIONAL INTELLIGENCE MONITORING SYSTEM ACTIVE



[USER ENTRY]:  need money


{'guidance': ['Your effort hasn’t failed — the market is slow, not your ability.', 'Pausing to regain clarity is not quitting.'], 'resources': [{'title': 'AICTE Internships', 'url': 'https://internship.aicte-india.org'}]}

============================== AI DECODING REPORT ==============================
User Input         : need money
Final Category    : CAREER (36.6% confidence)
---------------------------------------------------------------------------------
YOUTH EMOTIONAL DISTRIBUTION:
 - CAREER         : 100.0% ████████████████████ <--- CURRENT
---------------------------------------------------------------------------------
🧭 PERSONAL GUIDANCE:
You’re noticing your state early — that’s a healthy sign.
Pausing to regain clarity is not quitting.

🔗 RECOMMENDED RESOURCES:
 - AICTE Internships: https://internship.aicte-india.org
---------------------------------------------------------------------------------
ACTIVE POLICY ALERT: SIGNAL: Job market anxiety. Action: Increase vocational


[USER ENTRY]:  exit


In [21]:
import pandas as pd
from datetime import datetime
from transformers import pipeline
import warnings

# Clear warnings
warnings.filterwarnings("ignore")

# 1. LOAD AI MODELS
print("Initializing AI Models...")
# Using BART for topic understanding
topic_model = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
# Using DistilBERT for emotional tone
sentiment_model = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")
print("Models Loaded!")

# 2. LABELS
labels = ["academic stress", "career anxiety", "mental health", "physical wellness", "peace"]

# 3. PREDICTION FUNCTION
def get_prediction(text):
    # Topic
    t_res = topic_model(text, labels)
    topic = t_res['labels'][0]
    
    # Sentiment
    s_res = sentiment_model(text)[0]
    tone = s_res['label']
    
    # Response Logic
    if tone == "NEGATIVE":
        msg = f"It sounds like you are struggling with {topic}. Please take a breath."
    else:
        msg = f"I am glad you have a positive outlook on {topic}!"
        
    return topic, tone, msg

# 4. LOOP
while True:
    entry = input("\nEnter text: ")
    if entry.lower() == 'exit': break
    
    top, feel, advice = get_prediction(entry)
    print(f"TOPIC: {top}")
    print(f"SENTIMENT: {feel}")
    print(f"ADVICE: {advice}")

Initializing AI Models...


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

...


Enter text:  need money


TOPIC: career anxiety
SENTIMENT: NEGATIVE
ADVICE: It sounds like you are struggling with career anxiety. Please take a breath.



Enter text:  Feeling stuck


TOPIC: mental health
SENTIMENT: NEGATIVE
ADVICE: It sounds like you are struggling with mental health. Please take a breath.



Enter text:  exit


In [23]:
import pandas as pd
from datetime import datetime
from transformers import pipeline
import warnings

# Suppress warnings for a clean look
warnings.filterwarnings("ignore")

# ===================== 1. LOAD AI MODELS (THE BRAIN) =====================
print("⏳ Initializing YUVAPEACE AI Engine...")
topic_model = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
sentiment_model = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")
print("✅ System Ready!")

# ===================== 2. DATA CONFIGURATION =====================
# Mapping model labels to your specific categories
labels = ["academic stress", "career anxiety", "mental health", "physical health", "peace"]

policy_matrix = {
    "academic stress": "URGENT: Academic stress spike. Action: Deploy counseling to schools.",
    "career anxiety": "SIGNAL: Job market anxiety. Action: Increase vocational training.",
    "mental health": "CRITICAL: Emotional fatigue detected. Action: Activate mental health outreach.",
    "physical health": "VITAL: Physical health concerns. Action: Expand clinic access.",
    "peace": "STABILITY: Positive emotional state. Action: Support wellness programs."
}

resource_bank = {
    "career anxiety": {
        "resources": [{"title": "AICTE Internships", "url": "https://internship.aicte-india.org"}]
    },
    "mental health": {
        "resources": [{"title": "Kiran Mental Health Helpline", "url": "https://www.kiranhelpline.in"}]
    },
    "peace": {
        "resources": [{"title": "Mindfulness Guide", "url": "https://www.mindful.org"}]
    },
    "physical health": {
        "resources": [{"title": "eSanjeevani", "url": "https://esanjeevani.in"}]
    }
}



def get_prediction(user_text):
    # ML FIX: The hypothesis_template tells the AI to look for the 'struggle'
    t_res = topic_model(
        user_text, 
        labels, 
        hypothesis_template="This person is struggling with {}." 
    )
    topic = t_res['labels'][0]
    score = t_res['scores'][0]
    
    # Emotional Tone Detection
    s_res = sentiment_model(user_text)[0]
    tone = s_res['label']
    
    return topic, tone, score

    
# ===================== 3. MAIN SYSTEM LOOP =====================
db_history = []
print("\n🟢 YUVAPEACE: EMOTIONAL INTELLIGENCE MONITORING SYSTEM ACTIVE")

while True:
    user_input = input("\n[USER ENTRY]: ")
    if user_input.lower() == 'exit': break
    if not user_input.strip(): continue

    # --- STEP 1: AI PREDICTION ---
    t_res = topic_model(user_input, labels)
    topic = t_res['labels'][0]
    score = t_res['scores'][0]
    
    s_res = sentiment_model(user_input)[0]
    tone = s_res['label']

    # --- STEP 2: LOG DATA ---
    db_history.append({"Timestamp": datetime.now().strftime("%H:%M:%S"), "Topic": topic})
    df = pd.DataFrame(db_history)
    mindshare = df["Topic"].value_counts(normalize=True) * 100

    # --- STEP 3: FORMATTED OUTPUT ---
    print("\n" + "="*30 + " AI DECODING REPORT " + "="*30)
    print(f"User Input         : {user_input}")
    print(f"Final Category    : {topic.upper()} ({score*100:.1f}% confidence)")
    print("-" * 81)
    
    print("YOUTH EMOTIONAL DISTRIBUTION:")
    for cat, pct in mindshare.items():
        bar = "█" * int(pct / 5)
        mark = " <--- CURRENT" if cat == topic else ""
        print(f" - {cat.upper():<15}: {pct:>5.1f}% {bar}{mark}")

    print("-" * 81)
    
    # Dynamic Guidance based on Sentiment + Topic
    print("🧭 PERSONAL GUIDANCE:")
    if tone == "NEGATIVE":
        print(f"What you’re feeling regarding {topic} is heavy right now. You don’t have to solve everything today.")
    else:
        print(f"You’re noticing your state early regarding {topic} — that’s a healthy sign.")

    # Recommended Resources
    if topic in resource_bank:
        print("\n🔗 RECOMMENDED RESOURCES:")
        for r in resource_bank[topic]["resources"]:
            print(f" - {r['title']}: {r['url']}")

    print("-" * 81)
    print(f"ACTIVE POLICY ALERT: {policy_matrix.get(topic, 'Monitoring...')}")
    print("="*81)

⏳ Initializing YUVAPEACE AI Engine...


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

✅ System Ready!

🟢 YUVAPEACE: EMOTIONAL INTELLIGENCE MONITORING SYSTEM ACTIVE



[USER ENTRY]:  feeling stuck . Dont want to talk to anyone. Just want to be sit quiet. 



============================== AI DECODING REPORT ==============================
User Input         : feeling stuck . Dont want to talk to anyone. Just want to be sit quiet. 
Final Category    : PEACE (45.7% confidence)
---------------------------------------------------------------------------------
YOUTH EMOTIONAL DISTRIBUTION:
 - PEACE          : 100.0% ████████████████████ <--- CURRENT
---------------------------------------------------------------------------------
🧭 PERSONAL GUIDANCE:
What you’re feeling regarding peace is heavy right now. You don’t have to solve everything today.

🔗 RECOMMENDED RESOURCES:
 - Mindfulness Guide: https://www.mindful.org
---------------------------------------------------------------------------------
ACTIVE POLICY ALERT: STABILITY: Positive emotional state. Action: Support wellness programs.



[USER ENTRY]:  feeling stuck . Dont want to talk to anyone. Just want to be sit quiet.



============================== AI DECODING REPORT ==============================
User Input         : feeling stuck . Dont want to talk to anyone. Just want to be sit quiet.
Final Category    : PEACE (44.0% confidence)
---------------------------------------------------------------------------------
YOUTH EMOTIONAL DISTRIBUTION:
 - PEACE          : 100.0% ████████████████████ <--- CURRENT
---------------------------------------------------------------------------------
🧭 PERSONAL GUIDANCE:
What you’re feeling regarding peace is heavy right now. You don’t have to solve everything today.

🔗 RECOMMENDED RESOURCES:
 - Mindfulness Guide: https://www.mindful.org
---------------------------------------------------------------------------------
ACTIVE POLICY ALERT: STABILITY: Positive emotional state. Action: Support wellness programs.



[USER ENTRY]:  exit


In [24]:
import pandas as pd
import os
from datetime import datetime
from transformers import pipeline
import warnings

# Suppress warnings for a clean look
warnings.filterwarnings("ignore")

# ===================== 1. BACKEND INITIALIZATION =====================
DB_FILE = "yuvapeace_db.csv"

def load_backend():
    """Loads previous session data from CSV so your charts are historical."""
    if os.path.exists(DB_FILE):
        return pd.read_csv(DB_FILE).to_dict('records')
    return []

def save_to_backend(history):
    """Saves every entry to a CSV file for long-term monitoring."""
    pd.DataFrame(history).to_csv(DB_FILE, index=False)

# ===================== 2. LOAD AI MODELS (THE BRAIN) =====================
print("⏳ Initializing YUVAPEACE AI Engine...")
# BART handles the logic/context, DistilBERT handles the emotion
topic_model = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
sentiment_model = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")
db_history = load_backend()
print(f"✅ System Ready! (Loaded {len(db_history)} previous records)")

# ===================== 3. DATA CONFIGURATION =====================
labels = ["academic stress", "career anxiety", "mental health", "physical health", "peace"]

policy_matrix = {
    "academic stress": "URGENT: Academic stress spike. Action: Deploy counseling to schools.",
    "career anxiety": "SIGNAL: Job market anxiety. Action: Increase vocational training.",
    "mental health": "CRITICAL: Emotional fatigue detected. Action: Activate mental health outreach.",
    "physical health": "VITAL: Physical health concerns. Action: Expand clinic access.",
    "peace": "STABILITY: Positive emotional state. Action: Support wellness programs."
}

resource_bank = {
    "career anxiety": {"resources": [{"title": "AICTE Internships", "url": "https://internship.aicte-india.org"}]},
    "mental health": {"resources": [{"title": "Kiran Mental Health Helpline", "url": "https://www.kiranhelpline.in"}]},
    "peace": {"resources": [{"title": "Mindfulness Guide", "url": "https://www.mindful.org"}]},
    "physical health": {"resources": [{"title": "eSanjeevani", "url": "https://esanjeevani.in"}]}
}

# ===================== 4. THE REFINED AI ENGINE =====================
def get_prediction(user_text):
    """
    ML FIX: The hypothesis_template tells the AI to look for the 'struggle'.
    This prevents 'quiet' from being wrongly labeled as 'peace' when the tone is sad.
    """
    t_res = topic_model(
        user_text, 
        labels, 
        hypothesis_template="This person is experiencing {}." 
    )
    topic = t_res['labels'][0]
    score = t_res['scores'][0]
    
    # Emotional Tone Detection
    s_res = sentiment_model(user_text)[0]
    tone = s_res['label']
    
    return topic, tone, score

# ===================== 5. MAIN SYSTEM LOOP =====================
print("\n🟢 YUVAPEACE: EMOTIONAL INTELLIGENCE MONITORING SYSTEM ACTIVE")

while True:
    user_input = input("\n[USER ENTRY]: ")
    if user_input.lower() == 'exit': break
    if not user_input.strip(): continue

    # --- STEP 1: CALL THE PREDICTION FUNCTION ---
    topic, tone, score = get_prediction(user_input)

    # --- STEP 2: LOG DATA TO BACKEND ---
    db_history.append({
        "Timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"), 
        "Topic": topic,
        "Sentiment": tone,
        "Input": user_input
    })
    save_to_backend(db_history)

    # --- STEP 3: ANALYZE HISTORICAL TRENDS ---
    df = pd.DataFrame(db_history)
    mindshare = df["Topic"].value_counts(normalize=True) * 100

    # --- STEP 4: FORMATTED OUTPUT REPORT ---
    print("\n" + "="*30 + " AI DECODING REPORT " + "="*30)
    print(f"User Input         : {user_input}")
    print(f"Detected Category : {topic.upper()} ({score*100:.1f}% confidence)")
    print(f"Emotional Tone     : {tone}")
    print("-" * 81)
    
    print("YOUTH EMOTIONAL DISTRIBUTION (HISTORICAL DATA):")
    for cat, pct in mindshare.items():
        bar = "█" * int(pct / 5)
        mark = " <--- CURRENT" if cat == topic else ""
        print(f" - {cat.upper():<15}: {pct:>5.1f}% {bar}{mark}")

    print("-" * 81)
    print("🧭 PERSONAL GUIDANCE:")
    if tone == "NEGATIVE":
        print(f"It sounds like {topic} is weighing on you. It's okay to not be okay. Take small steps.")
    else:
        print(f"Your perspective on {topic} is positive! Keeping this mindset is a great strength.")

    if topic in resource_bank:
        print("\n🔗 RECOMMENDED RESOURCES:")
        for r in resource_bank[topic]["resources"]:
            print(f" - {r['title']}: {r['url']}")

    print("-" * 81)
    
    # SMART POLICY ALERT: Adjusts based on Sentiment
    if tone == "POSITIVE":
        print(f"ACTIVE POLICY ALERT: STABILITY: Positive trends in {topic}. Action: Maintain current support.")
    else:
        print(f"ACTIVE POLICY ALERT: {policy_matrix.get(topic, 'Monitoring...')}")
    print("="*81)

⏳ Initializing YUVAPEACE AI Engine...


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

✅ System Ready! (Loaded 0 previous records)

🟢 YUVAPEACE: EMOTIONAL INTELLIGENCE MONITORING SYSTEM ACTIVE



[USER ENTRY]:  feeling stuck . Dont want to talk to anyone. Just want to be sit quiet.


...


[USER ENTRY]:  need money



============================== AI DECODING REPORT ==============================
User Input         : need money
Detected Category : MENTAL HEALTH (49.8% confidence)
Emotional Tone     : NEGATIVE
---------------------------------------------------------------------------------
YOUTH EMOTIONAL DISTRIBUTION (HISTORICAL DATA):
 - MENTAL HEALTH  : 100.0% ████████████████████ <--- CURRENT
---------------------------------------------------------------------------------
🧭 PERSONAL GUIDANCE:
It sounds like mental health is weighing on you. It's okay to not be okay. Take small steps.

🔗 RECOMMENDED RESOURCES:
 - Kiran Mental Health Helpline: https://www.kiranhelpline.in
---------------------------------------------------------------------------------
ACTIVE POLICY ALERT: CRITICAL: Emotional fatigue detected. Action: Activate mental health outreach.



[USER ENTRY]:  want internships



============================== AI DECODING REPORT ==============================
User Input         : want internships
Detected Category : ACADEMIC STRESS (34.9% confidence)
Emotional Tone     : NEGATIVE
---------------------------------------------------------------------------------
YOUTH EMOTIONAL DISTRIBUTION (HISTORICAL DATA):
 - MENTAL HEALTH  :  83.3% ████████████████
 - ACADEMIC STRESS:  16.7% ███ <--- CURRENT
---------------------------------------------------------------------------------
🧭 PERSONAL GUIDANCE:
It sounds like academic stress is weighing on you. It's okay to not be okay. Take small steps.
---------------------------------------------------------------------------------
ACTIVE POLICY ALERT: URGENT: Academic stress spike. Action: Deploy counseling to schools.



[USER ENTRY]:  exit


In [25]:
import pandas as pd
import os
from datetime import datetime
from transformers import pipeline
import warnings

warnings.filterwarnings("ignore")

# ===================== 1. AI ENGINE =====================
print("⏳ Initializing YUVAPEACE AI... (Please wait a moment)")
# We use BART-Large because it is the most accurate for Zero-Shot
topic_model = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
sentiment_model = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

# ===================== 2. ACCURATE CONFIGURATION =====================
# These labels are semantically distinct so the AI doesn't get confused
labels = ["academic pressure", "career & money", "mental health", "physical health", "wellness & peace"]

policy_matrix = {
    "academic pressure": "POLICY: Academic stress detected. Action: Review exam schedules.",
    "career & money": "POLICY: Economic anxiety detected. Action: Promote internship portal.",
    "mental health": "POLICY: Emotional fatigue detected. Action: Connect to Kiran Helpline.",
    "physical health": "POLICY: Health concern detected. Action: Suggest eSanjeevani checkup.",
    "wellness & peace": "POLICY: Positive stability. Action: Maintain community programs."
}

# ===================== 3. THE LOGIC FUNCTION =====================
def get_prediction(user_text):
    # We use a neutral template so the AI isn't "biased" or "hardcoded"
    t_res = topic_model(user_text, labels, hypothesis_template="This is about {}.")
    
    topic = t_res['labels'][0]
    score = t_res['scores'][0]
    
    s_res = sentiment_model(user_text)[0]
    tone = s_res['label'] # POSITIVE or NEGATIVE
    
    return topic, tone, score

# ===================== 4. EXECUTION LOOP =====================
db_history = []
print("✅ System Ready! Type 'exit' to stop.\n")

while True:
    user_input = input("[USER ENTRY]: ")
    if user_input.lower() == 'exit': break
    if not user_input.strip(): continue

    # Calling the AI
    topic, tone, score = get_prediction(user_input)

    # Logging
    db_history.append({"Topic": topic, "Tone": tone})
    df = pd.DataFrame(db_history)
    stats = df["Topic"].value_counts(normalize=True) * 100

    # Output Report
    print(f"\n" + "="*20 + " AI ANALYSIS " + "="*20)
    print(f"INPUT    : {user_input}")
    print(f"CATEGORY : {topic.upper()} ({score*100:.1f}% confidence)")
    print(f"SENTIMENT: {tone}")
    print("-" * 53)
    
    # Historical Trend
    print("CURRENT TRENDS:")
    for cat, pct in stats.items():
        print(f" - {cat:<18}: {pct:>5.1f}% {'█' * int(pct/10)}")

    print("-" * 53)
    print(f"ACTION: {policy_matrix.get(topic)}")
    print("="*53 + "\n")

⏳ Initializing YUVAPEACE AI... (Please wait a moment)


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

✅ System Ready! Type 'exit' to stop.



[USER ENTRY]:  want internships and peace



==================== AI ANALYSIS ====================
INPUT    : want internships and peace
CATEGORY : WELLNESS & PEACE (68.1% confidence)
SENTIMENT: POSITIVE
-----------------------------------------------------
CURRENT TRENDS:
 - wellness & peace  : 100.0% ██████████
-----------------------------------------------------
ACTION: POLICY: Positive stability. Action: Maintain community programs.



[USER ENTRY]:  exit


In [ ]:
import pandas as pd
from datetime import datetime
from transformers import pipeline
import warnings

warnings.filterwarnings("ignore")

# ===================== 1. THE BRAIN (NO HARDCODING) =====================
print("⏳ Loading Large-Scale AI Models...")
# We use the 'large' model (400MB+) for actual reasoning, not just keyword matching
topic_model = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
sentiment_model = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

# ===================== 2. SEMANTIC LABELS =====================
# These are broad "buckets" that the AI understands naturally
labels = ["education", "career and jobs", "emotional health", "physical wellness", "peace and stability"]

# Policy logic is now based on the detected emotion, not just the word
policy_alerts = {
    "NEGATIVE": "ACTION REQUIRED: Deploy support resources for ",
    "POSITIVE": "STABILITY DETECTED: Support continued growth in "
}

# ===================== 3. THE PREDICTION LOGIC =====================
def get_prediction(user_text):
    # Pure ML: No biased hypothesis template. Let the model decide.
    t_res = topic_model(user_text, labels, multi_label=True)
    
    # Extract highest scoring category
    topic = t_res['labels'][0]
    score = t_res['scores'][0]
    
    # Extract sentiment
    s_res = sentiment_model(user_text)[0]
    tone = s_res['label']
    
    return topic, tone, score

# ===================== 4. SYSTEM EXECUTION =====================
db_history = []
print("✅ System Active. Ready for accurate analysis.\n")

while True:
    user_input = input("[USER ENTRY]: ")
    if user_input.lower() == 'exit': break
    if not user_input.strip(): continue

    # 1. AI Analysis
    topic, tone, score = get_prediction(user_input)

    # 2. Database Logging
    db_history.append({"Topic": topic, "Tone": tone})
    df = pd.DataFrame(db_history)
    stats = df["Topic"].value_counts(normalize=True) * 100

    # 3. Dynamic Output
    print(f"\n" + "—"*30 + " AI DECODING " + "—"*30)
    print(f"INPUT     : {user_input}")
    print(f"INTENT    : {topic.upper()} ({score*100:.1f}% confidence)")
    print(f"EMOTION   : {tone}")
    print("-" * 73)
    
    # Historical Visualization
    print("CURRENT TRENDSHARE:")
    for cat, pct in stats.items():
        bar = "█" * int(pct/5)
        print(f" - {cat:<20}: {pct:>5.1f}% {bar}")

    print("-" * 73)
    
    # Dynamic Policy Generation (Not hardcoded to topic alone)
    alert_prefix = policy_alerts.get(tone)
    print(f"POLICY: {alert_prefix}{topic.upper()}")
    print("—"*73 + "\n")

⏳ Loading Large-Scale AI Models...


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

✅ System Active. Ready for accurate analysis.



[USER ENTRY]:  want internship now and a peace in life



—————————————————————————————— AI DECODING ——————————————————————————————
INPUT     : want internship now and a peace in life
INTENT    : CAREER AND JOBS (92.8% confidence)
EMOTION   : POSITIVE
-------------------------------------------------------------------------
CURRENT TRENDSHARE:
 - career and jobs     : 100.0% ████████████████████
-------------------------------------------------------------------------
POLICY: STABILITY DETECTED: Support continued growth in CAREER AND JOBS
—————————————————————————————————————————————————————————————————————————



[USER ENTRY]:  feeling low , stuck , sad 



—————————————————————————————— AI DECODING ——————————————————————————————
INPUT     : feeling low , stuck , sad 
INTENT    : EMOTIONAL HEALTH (93.9% confidence)
EMOTION   : NEGATIVE
-------------------------------------------------------------------------
CURRENT TRENDSHARE:
 - career and jobs     :  50.0% ██████████
 - emotional health    :  50.0% ██████████
-------------------------------------------------------------------------
POLICY: ACTION REQUIRED: Deploy support resources for EMOTIONAL HEALTH
—————————————————————————————————————————————————————————————————————————



[USER ENTRY]:  looking for jobs



—————————————————————————————— AI DECODING ——————————————————————————————
INPUT     : looking for jobs
INTENT    : CAREER AND JOBS (97.3% confidence)
EMOTION   : NEGATIVE
-------------------------------------------------------------------------
CURRENT TRENDSHARE:
 - career and jobs     :  66.7% █████████████
 - emotional health    :  33.3% ██████
-------------------------------------------------------------------------
POLICY: ACTION REQUIRED: Deploy support resources for CAREER AND JOBS
—————————————————————————————————————————————————————————————————————————

